In [23]:
import cv2
import os
import glob
import torch
from pathlib import Path
from ultralytics import YOLO

In [ ]:
BASE_DIR = Path.cwd()

# 네 종목 폴더
EXERCISE_DIRS = {
    "leg": BASE_DIR / "custom_new" / "leg",
    "lunge": BASE_DIR / "custom_new" / "lunge",
    "plank": BASE_DIR / "custom_new" / "plank",
    "pushup": BASE_DIR / "custom_new" / "pushup",
    "plank" : BASE_DIR / "custom_new" / "plank_new"
}

capture_sec = 0.1  # 캡처 간격 (0.5초->0.1초 변경)

# 저장 폴더
img_dir = BASE_DIR / "dataset-cf" / "images" / "train"
txt_dir = BASE_DIR / "dataset-cf" / "labels" / "train"

img_dir.mkdir(parents=True, exist_ok=True)
txt_dir.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = YOLO(str(BASE_DIR / "yolov8x-pose.pt")).to(device)

In [25]:
total_saved_count = 0

for exercise_name, input_video_dir in EXERCISE_DIRS.items():

    video_files = glob.glob(os.path.join(str(input_video_dir), "*.mp4"))
    video_files += glob.glob(os.path.join(str(input_video_dir), "*.mov"))

    if not video_files:
        print(f"{exercise_name}: 영상 없음")
        continue

    print(f"\n[{exercise_name}] 총 {len(video_files)}개의 영상")

    for video_path in video_files:
        base_name = os.path.basename(video_path)
        output_name = os.path.splitext(base_name)[0]

        print(f"처리 중인 영상: {base_name}")

        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)

        frame_interval = max(1, int(fps * capture_sec))

        frame_count = 0
        saved_count_per_video = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            if frame_count % frame_interval == 0:

                results = model.predict(
                    source=frame,
                    imgsz=1280,
                    conf=0.40,
                    iou=0.8,
                    verbose=False
                )

                r = results[0]

                if r.boxes is None or len(r.boxes) == 0:
                    frame_count += 1
                    continue

                if r.keypoints is None:
                    frame_count += 1
                    continue

                # 운동명 추가
                img_filename = (
                    f"{exercise_name}_{output_name}_{saved_count_per_video:04d}.jpg"
                )

                img_path = os.path.join(img_dir, img_filename)
                txt_path = os.path.join(
                    txt_dir,
                    img_filename.replace(".jpg", ".txt")
                )

                cv2.imwrite(img_path, frame)

                with open(txt_path, "w") as f:
                    for i in range(len(r.boxes)):
                        box = r.boxes[i].xywhn[0].tolist()
                        kpts = r.keypoints[i].xyn[0].tolist()
                        confs = r.keypoints[i].conf[0].tolist()

                        line = (
                            f"0 {box[0]:.6f} {box[1]:.6f} "
                            f"{box[2]:.6f} {box[3]:.6f} "
                        )

                        for kpt, conf in zip(kpts, confs):
                            if conf > 0.1:
                                line += (
                                    f"{kpt[0]:.6f} "
                                    f"{kpt[1]:.6f} 2 "
                                )
                            else:
                                line += "0.000000 0.000000 0 "

                        f.write(line.strip() + "\n")

                saved_count_per_video += 1
                total_saved_count += 1

            frame_count += 1

        cap.release()

        print(
            f"완료: {output_name} 에서 "
            f"{saved_count_per_video}장 추출됨."
        )

print(f"\n총 학습 데이터: {total_saved_count}장")


[plank] 총 3개의 영상
처리 중인 영상: plank_09.mp4
완료: plank_09 에서 359장 추출됨.
처리 중인 영상: plank_10.mp4
완료: plank_10 에서 347장 추출됨.
처리 중인 영상: plank_11.mp4
완료: plank_11 에서 115장 추출됨.

총 학습 데이터: 821장
